# AETHER — Peptide-MHC Dataset Overview

Describes the MHCAttnNet class-I binding data in `data/mhcattnnet/`, and surfaces
the three properties that drive the training design:

1. **Class imbalance** (~77% binders) — accuracy is a weak measure here, so AUROC
   is what gets reported and what selects checkpoints.
2. **MHC redundancy** (~150 unique MHC sequences carry ~344k rows) — which is why
   training precomputes the MHC embeddings once instead of re-encoding them every
   step.
3. **Long-tailed allele coverage** — a pooled metric is dominated by a handful of
   alleles, so results are also broken down per allele.

Each cell produces one standalone 600 dpi PNG in this folder.

In [ ]:
from pathlib import Path
import json
import logging
import os
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

# matplotlib reports a missing font through the logging module rather than
# warnings, so filterwarnings alone never silences "findfont: Font family
# 'Arial' not found".
warnings.filterwarnings("ignore", message=".*findfont.*")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# ---------------------------------------------------------------------
# plot style
# ---------------------------------------------------------------------
# Arial is the house font, but it ships with no Linux distribution and is not
# installed on this machine. Declaring a fallback chain means the figures use
# Arial wherever it exists and a metrically similar sans elsewhere, rather than
# matplotlib silently substituting whatever it has. Liberation Sans and Nimbus
# Sans share Arial's metrics, so text occupies the same space; DejaVu Sans is
# matplotlib's bundled last resort and is slightly wider.
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = [
    "Arial", "Liberation Sans", "Nimbus Sans", "Helvetica", "DejaVu Sans",
]
mpl.rcParams["font.size"] = 10
DPI = 600
FIGSIZE = (6, 3.8)   # every figure stands alone; no subplots

BAR_COLOR = "#4b5563"
LIGHT_BAR_COLOR = "#d1d5db"
REF_LINE = "#6b7280"
GRID_COLOR = "#f0f0f0"

# ink scale for text that is not an axis label
INK = "#111827"
INK_SOFT = "#4b5563"
INK_MUTED = "#6b7280"

TAB = {"blue": "#4e79a7", "orange": "#f28e2b", "red": "#e15759", "teal": "#76b7b2",
       "green": "#59a14f", "yellow": "#edc948", "purple": "#b07aa1",
       "pink": "#ff9da7", "brown": "#9c755f", "gray": "#bab0ac"}

# roles, so a plot is written against what the color means
BINDER, NON_BINDER = TAB["blue"], TAB["orange"]
TRAIN_COLOR, VAL_COLOR = TAB["blue"], TAB["red"]
CLASS_COLORS = {"Binder": BINDER, "Non-binder": NON_BINDER}


def new_fig(figsize=FIGSIZE):
    """one standalone figure with its axes styled"""
    fig, ax = plt.subplots(figsize=figsize)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color=GRID_COLOR, linewidth=0.6)
    ax.set_axisbelow(True)
    return fig, ax


def save_fig(fig, name):
    """600 dpi png into figures/, cropped to everything drawn"""
    path = FIG_DIR / name
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    print(f"saved {path}")


def left_legend(ax, handles, title, anchor):
    """left-aligned legend outside the axes; call again to stack another one below"""
    if ax.get_legend() is not None:
        previous = ax.get_legend()
        ax.add_artist(previous)
        previous.set_clip_on(False)
    legend = ax.legend(handles=handles, title=title, frameon=False,
                       bbox_to_anchor=anchor, loc="upper left")
    legend._legend_box.align = "left"
    legend.get_title().set_ha("left")
    return legend


def class_handles():
    return [Patch(facecolor=c, label=k) for k, c in CLASS_COLORS.items()]


# ---------------------------------------------------------------------
# paths, relative to this notebook
# ---------------------------------------------------------------------
FIG_DIR = Path.cwd()
REPO = FIG_DIR
while REPO != REPO.parent and not (REPO / "data").is_dir():
    REPO = REPO.parent
print("repo:", REPO)

In [ ]:
DATA = REPO / "data" / "mhcattnnet" / "classI"

splits = {name: pd.read_csv(DATA / f"{name}.csv") for name in ("train", "val", "test")}
for df in splits.values():
    df["pep_len"] = df["peptide"].str.len()
    df["mhc_len"] = df["mhc_amino_acid"].str.len()

train = splits["train"]

pd.DataFrame({
    name: {
        "Rows":            len(df),
        "Binders":         int(df["bind"].sum()),
        "Positive Rate":   round(df["bind"].mean(), 3),
        "Unique Peptides": df["peptide"].nunique(),
        "Unique MHC Seqs": df["mhc_amino_acid"].nunique(),
        "Unique Alleles":  df["mhc_allele"].nunique(),
    }
    for name, df in splits.items()
}).T

## Class Balance

A constant "binder" prediction scores 0.774 accuracy on this data. That is the
number any model has to beat before accuracy means anything, which is why
`train.py` selects `best.pt` on AUROC instead.

In [ ]:
fig, ax = new_fig((6, 2.8))
ax.grid(False)

names = list(splits)
pos = np.array([splits[n]["bind"].mean() for n in names])
y = np.arange(len(names))

ax.barh(y, pos, height=0.5, color=BINDER)
ax.barh(y, 1 - pos, height=0.5, left=pos, color=NON_BINDER)
for i, p in enumerate(pos):
    ax.text(p - 0.012, i, f"{p:.1%}", ha="right", va="center", color="white", fontsize=9)
    ax.text(p + 0.012, i, f"{1 - p:.1%}", ha="left", va="center", color="white", fontsize=9)

ax.set_yticks(y, [f"{n}\n{len(splits[n]):,} rows" for n in names])
ax.set_xlim(0, 1)
ax.set_xlabel("Share of Rows")
ax.invert_yaxis()
ax.set_title("Binders Outnumber Non-Binders ~3:1 in Every Split")
left_legend(ax, class_handles(), "", (1.02, 1.0))

save_fig(fig, "dataset_class_balance.png")
plt.show()

## Peptide Length

Class-I peptides cluster hard at 9 residues, the canonical length for the closed
class-I binding groove.

Grouped bars rather than overlaid histograms: two translucent fills on the same
axes read as a stack, which invites the wrong comparison.

In [ ]:
def length_bin(n):
    return "≤7" if n <= 7 else (str(n) if n <= 12 else ("13-25" if n <= 25 else ">25"))


LENGTH_ORDER = ["≤7", "8", "9", "10", "11", "12", "13-25", ">25"]
train["len_bin"] = train["pep_len"].map(length_bin)

share = (train.groupby(["len_bin", "bind"]).size().unstack(fill_value=0)
         .reindex(LENGTH_ORDER).fillna(0))
share = share / share.sum()          # within-class proportion

fig, ax = new_fig()
x = np.arange(len(LENGTH_ORDER))
ax.bar(x - 0.19, share[1], width=0.36, color=BINDER)
ax.bar(x + 0.19, share[0], width=0.36, color=NON_BINDER)
ax.set_xticks(x, LENGTH_ORDER)
ax.set_xlabel("Peptide Length (Residues)")
ax.set_ylabel("Share of Class")
ax.set_title("Peptide Length Distribution by Class")
left_legend(ax, class_handles(), "", (1.02, 1.0))

share_9 = (train["pep_len"] == 9).mean()
ax.text(0.98, 0.86, f"{share_9:.0%} of all rows are 9-mers", transform=ax.transAxes,
        ha="right", fontsize=9, color=REF_LINE)

save_fig(fig, "dataset_peptide_length.png")
plt.show()

In [ ]:
# Does label balance shift off the 9-mer peak?
by_len = (train[train["pep_len"].between(8, 15)]
          .groupby("pep_len").agg(rate=("bind", "mean"), n=("bind", "size")).reset_index())
overall = train["bind"].mean()

fig, ax = new_fig()
ax.bar(by_len["pep_len"], by_len["rate"], width=0.62, color=BAR_COLOR)
ax.axhline(overall, color=REF_LINE, linestyle="--", linewidth=0.8)
ax.text(7.6, overall + 0.025, f"overall {overall:.1%}", va="bottom", ha="left",
        fontsize=9, color=REF_LINE)
for _, r in by_len.iterrows():
    ax.text(r["pep_len"], r["rate"] + 0.02, f"{r['rate']:.0%}", ha="center",
            va="bottom", fontsize=8)
ax.set_xlabel("Peptide Length (Residues)")
ax.set_ylabel("Fraction Labelled Binder")
ax.set_ylim(0, 1.12)
ax.set_title("Binding Rate Is Roughly Flat across Length")

save_fig(fig, "dataset_bind_rate_by_length.png")
plt.show()

## MHC Redundancy — the Fact That Shapes the Training Loop

The MHC side of each row is a full ~275-residue allele sequence, about 30x longer
than the peptide. But there are only ~150 distinct ones across ~344k rows, so
encoding them row-by-row runs the same forward pass thousands of times for
nothing.

AETHER runs the frozen backbone once over the unique MHC sequences and pins the
result (`--precompute_mhc`, on by default). The next two figures are the argument.

In [ ]:
# How many rows each distinct value has to serve
uniq = np.array([train["mhc_amino_acid"].nunique(), train["peptide"].nunique()])
reuse = len(train) / uniq
labels = [f"MHC Sequences\n{uniq[0]:,} distinct", f"Peptides\n{uniq[1]:,} distinct"]

fig, ax = new_fig((5, 3.8))
ax.bar(np.arange(2), reuse, width=0.46, color=[TAB["blue"], LIGHT_BAR_COLOR])
for i, r in enumerate(reuse):
    ax.text(i, r * 1.35, f"{r:,.0f}x", ha="center", fontsize=13, color=BAR_COLOR)
ax.set_yscale("log")
ax.set_ylim(1, reuse.max() * 6)
ax.set_xticks(np.arange(2), labels)
ax.set_xlim(-0.6, 1.6)
ax.set_ylabel("Rows per Distinct Value (log)")
ax.set_title("The MHC Side Is ~2,000x Redundant")

save_fig(fig, "dataset_mhc_redundancy.png")
plt.show()

In [ ]:
# Distribution of that reuse across MHC sequences
reps = train.groupby("mhc_amino_acid").size()

fig, ax = new_fig()
ax.hist(reps, bins=40, color=TAB["teal"])
ax.axvline(reps.median(), color=REF_LINE, linestyle="--", linewidth=0.8)
ax.text(reps.median() + reps.max() * 0.02, ax.get_ylim()[1] * 0.9,
        f"median {reps.median():,.0f} rows", fontsize=9, color=REF_LINE)
ax.set_xlabel("Training Rows Sharing One MHC Sequence")
ax.set_ylabel("Number of MHC Sequences")
ax.set_title("Each MHC Sequence Is Re-Used Thousands of Times")

save_fig(fig, "dataset_mhc_reuse_distribution.png")
plt.show()

## Allele Coverage

Rows are not spread evenly over alleles. A pooled AUROC is dominated by the
well-represented ones, so the results notebook breaks performance down per allele.

In [ ]:
top = (train.groupby("mhc_allele").agg(n=("bind", "size"), rate=("bind", "mean"))
       .sort_values("n", ascending=False).head(20).iloc[::-1])

fig, ax = new_fig((6.5, 5.6))
ax.barh(np.arange(len(top)), top["n"], height=0.66, color=BAR_COLOR)
for i, (n, rate) in enumerate(zip(top["n"], top["rate"])):
    ax.text(n + len(train) * 0.005, i, f"{n:,} ({rate:.0%} bind)", va="center", fontsize=8)
ax.set_yticks(np.arange(len(top)), top.index, fontsize=8.5)
ax.set_xlabel("Training Rows")
ax.set_xlim(0, top["n"].max() * 1.3)
cov = top["n"].sum() / len(train)
ax.set_title(f"Top 20 Alleles Cover {cov:.0%} of Training Rows")

save_fig(fig, "dataset_allele_coverage.png")
plt.show()

In [ ]:
# The long tail, stated cumulatively
counts = train.groupby("mhc_allele").size().sort_values(ascending=False)
cum = counts.cumsum() / counts.sum()
xs = np.arange(1, len(cum) + 1)

fig, ax = new_fig()
ax.plot(xs, cum.values, color=TAB["blue"], linewidth=1.6)
ax.fill_between(xs, cum.values, color=TAB["blue"], alpha=0.12)
for frac in (0.5, 0.9):
    k = int((cum < frac).sum()) + 1
    ax.plot([k, k], [0, frac], color=REF_LINE, linestyle=":", linewidth=0.8)
    ax.annotate(f"{k} alleles → {frac:.0%} of rows", xy=(k, frac),
                xytext=(k + 12, frac - 0.12), fontsize=9, color=REF_LINE,
                arrowprops=dict(arrowstyle="-", color=REF_LINE, linewidth=0.8))
ax.set_xlabel("Alleles, Ranked by Support")
ax.set_ylabel("Cumulative Share of Rows")
ax.set_ylim(0, 1.02)
ax.set_xlim(1, len(cum))
ax.set_title("Allele Support Is Long-Tailed")

save_fig(fig, "dataset_allele_concentration.png")
plt.show()

## Takeaways for Training

| Observation | Consequence in `train.py` |
|---|---|
| 77% of rows are binders | report **AUROC / AUPRC**, select `best.pt` on AUROC (`--best_metric auc`) |
| ~150 unique MHC sequences carry ~344k rows | **precompute MHC embeddings once** (`--precompute_mhc`), the default |
| MHC sequence is ~30x longer than the peptide | precompute removes nearly all backbone cost; peptides stay encoded live |
| Allele support is long-tailed | break test performance down **per allele**, not just pooled |
| 9-mers dominate | off-length peptides are a thin slice — check them separately before trusting a pooled gain |